In [1]:
from sedona.spark import *
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".
26/03/16 11:50:46 INFO core/src/lib.rs: Sedona native acceleration engine v0.12.0 ready
                                                                                

In [2]:
database = 'gde_silver'

In [3]:
sedona.sql("""
SELECT COUNT(*) 
FROM org_catalog.gde_bronze.king_co_homes 
WHERE geometry IS NOT NULL
""").show()

[Stage 3:==================================================>        (6 + 1) / 7]

+--------+
|count(1)|
+--------+
|  591513|
+--------+



In [4]:
# Seismic Hazards Table

sedona.sql(f'''
update org_catalog.gde_bronze.king_co_homes
set geometry = st_point(longitude, latitude) 
''')

DataFrame[]

In [5]:
# Seismic Hazards Table

sedona.sql(f'''
create or replace table org_catalog.{database}.homes_seismic_hazards as
select                                                                                                      
a.sale_id,                                                                                                  
b.OBJECTID as hazard_id,                                                                                                  
b.HAZARD as hazard                                                                                                     
from org_catalog.gde_bronze.king_co_homes a 
join org_catalog.gde_bronze.seismic_hazards_bronze b  
on ST_Intersects(a.geometry, b.geometry)
''')

DataFrame[]

In [6]:
# Flood Hazards Table

sedona.sql(f'''
create or replace table org_catalog.{database}.homes_flood_hazards as
    select                                                                                                      
        a.sale_id,                                                                                                  
        b.FLD_AR_ID as flood_zone_id,                                                                                                  
        b.FLD_ZONE as flood_zone                                                                                                     
    from org_catalog.gde_bronze.king_co_homes a 
    join org_catalog.gde_bronze.fema_flood_zones_bronze b  
        on st_contains(b.geometry, a.geometry)
''')

DataFrame[]

In [7]:
# School Achievement Table
sedona.sql(f'''
create or replace table org_catalog.gde_silver.homes_school_scores as
select                                                                                                      
    a.sale_id,                                                                                                  
    avg(b.high_achievement_percent) as avg_school_high_achievement                                                                                                   
from org_catalog.gde_bronze.king_co_homes a 
join org_catalog.gde_silver.school_data b  
    on st_knn(a.geometry, b.geometry, 5)
where b.high_achievement_percent IS NOT NULL
group by a.sale_id
''')

sedona.sql(f'''
create or replace table org_catalog.{database}.homes_school_scores as
    select                                                                                                      
        a.sale_id,                                                                                                  
        avg(b.high_achievement_percent) as avg_school_high_achievement                                                                                                   
    from org_catalog.gde_bronze.king_co_homes a 
    join org_catalog.{database}.school_data b  
        on st_knn(a.geometry, b.geometry, 5)
    group by a.sale_id
''')

26/03/16 11:51:20 WARN JoinQueryDetector: Filter pushdown detected on the object side of a KNN join. This may cause the KNN join to return incorrect results. Consider materializing the object side before the join to prevent filter pushdown.
                                                                                

DataFrame[]

In [8]:
sedona.sql("SELECT COUNT(*) FROM org_catalog.gde_silver.homes_school_scores").show()

+--------+
|count(1)|
+--------+
|  591513|
+--------+



In [9]:
sedona.sql("""
SELECT 
    COUNT(*) as total,
    COUNT(high_achievement_percent) as non_null,
    SUM(CASE WHEN high_achievement_percent IS NULL THEN 1 ELSE 0 END) as nulls
FROM org_catalog.gde_silver.school_data
""").show()

+-----+--------+-----+
|total|non_null|nulls|
+-----+--------+-----+
|  900|     842|   58|
+-----+--------+-----+



In [10]:
sedona.sql("DESCRIBE org_catalog.gde_bronze.schools_bronze").show(50, truncate=False)

+---------------------+---------+-------+
|col_name             |data_type|comment|
+---------------------+---------+-------+
|geometry             |geometry |NULL   |
|AYPCode              |string   |NULL   |
|CongressionalDistrict|string   |NULL   |
|County               |string   |NULL   |
|ESDCode              |bigint   |NULL   |
|ESDName              |string   |NULL   |
|Email                |string   |NULL   |
|GeoCoded_X           |double   |NULL   |
|GeoCoded_Y           |double   |NULL   |
|GradeCategory        |string   |NULL   |
|HighestGrade         |string   |NULL   |
|LEACode              |bigint   |NULL   |
|LEAName              |string   |NULL   |
|LegislativeDistrict  |string   |NULL   |
|LowestGrade          |string   |NULL   |
|MailingAddress       |string   |NULL   |
|NCES_X               |double   |NULL   |
|NCES_Y               |double   |NULL   |
|Phone                |string   |NULL   |
|Principal            |string   |NULL   |
|School               |string   |N